<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_01_loss_functions_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 01 — Where the Loss Comes From

**Paired with L6.1 · Loss Functions and Gradients**

The recipe of L6.1: **write down the probability of the data given the model's
output, take the logarithm, put a minus sign in front. That is the loss.** You
will

1. derive the mean squared error from Gaussian noise, and check that the two
   have the same minimiser;
2. implement the cross entropy in NumPy and check it against
   `nn.CrossEntropyLoss`;
3. train one classifier under both losses and compare;
4. corrupt one reading and watch the fitted line move under each loss.

---

## 0 · Setup

**What the three cells below do.** The first fetches the library file
`Ex_6_core.py` when you run on Colab. The second keeps what this notebook saves
in your Google Drive, so the report notebook can read it later. The third loads
the data of sections 1 and 4.

**The data: a load cell.** A load cell is a force sensor, the part inside a
weighing scale that turns a weight into an electrical signal. To **calibrate**
one, you apply known loads $x$ and record the readings $y$. The sensor should
follow a straight line, $y = a x + b$: the slope $a$ is its sensitivity and the
intercept $b$ its zero offset. Every reading also carries a small random error,
the **noise**. Its size, $\sigma = 0.35$, is a property of the instrument and is
known from its calibration. The forty readings here are simulated, so the true
$a$, $b$ and $\sigma$ are known and every fit can be scored against them.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()


In [ ]:
# A load cell, calibrated. Forty readings of a straight line plus noise.
#   core.calibration_dataset(n, seed, sigma) -> x, y, each (n,)
#   core.TRUE_SLOPE, core.TRUE_INTERCEPT, core.TRUE_SIGMA are what
#   generated them, so every fit below can be scored against the truth.
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)

x, y = core.calibration_dataset(n=40)
print("calibration points:", x.shape)
print("true slope %.2f, intercept %.2f, noise sigma %.2f"
      % (core.TRUE_SLOPE, core.TRUE_INTERCEPT, core.TRUE_SIGMA))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(x, core.TRUE_SLOPE * x + core.TRUE_INTERCEPT, lw=1.6, ls="--",
        color="#888888", label="true calibration")
ax.plot(x, y, "o", ms=6, color="#111111", label="readings")
ax.set_xlabel("applied load [normalised]"); ax.set_ylabel("sensor reading")
ax.set_title("Forty calibration points from a load cell")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.** Forty points scattered about a dashed line of slope 2.4
and intercept 0.8. The load cell's error is Gaussian, independent between
readings and of constant variance: exactly what the mean squared error assumes.

---

## 1 · From a Gaussian assumption to the mean squared error

**The goal.** Least squares, "choose the line with the smallest squared
errors", is usually taught as a rule that works. This section shows where it
comes from. Assume the sensor's errors follow a Gaussian (bell-shaped)
distribution, write down the probability of the forty readings, and minimise
minus its logarithm, the **negative log likelihood (NLL)**. The result is the
mean squared error (MSE) plus a constant, so the two pick the same line.

**How it is checked.** Nothing is solved at first. Every candidate line is
tried instead: 601 slopes $a$ from 1.8 to 3.0 against 601 intercepts $b$ from
0.2 to 1.4, which is 361,201 lines. Both losses are computed for each line, and
for each loss the line with the lowest value is found. The two answers are then
compared with the exact least-squares solution and with the true line.

**Reading the contour plots.** Each point on the map is one candidate line,
its intercept across and its slope up. Each ring joins lines with the same
loss, like the height lines on a map, and the centre of the rings is the best
line. The two panels have the same rings around the same centre, which is the
claim of this section in a picture. $\sigma$ appears in the NLL but does not
move the centre.

This is not a fitting exercise. The question is *why least squares?* Write the
model as $\hat{y}(x) = a x + b$ and the assumption as

$$y_i = \hat{y}(x_i) + \varepsilon_i, \qquad
\varepsilon_i \sim \mathcal{N}(0, \sigma^2), \quad \text{independent.}$$

The density of one reading is

$$p(y_i \mid a, b) = \frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left(-\frac{(y_i - \hat{y}(x_i))^2}{2\sigma^2}\right),$$

the density of the data is the product over $i$, and its logarithm with a minus
sign in front is the **negative log likelihood**

$$\mathcal{L}_{\mathrm{NLL}}(a, b) \;=\;
\underbrace{\frac{n}{2}\log(2\pi\sigma^2)}_{\text{does not contain } a, b}
\;+\; \frac{1}{2\sigma^2}\sum_{i=1}^{n}\bigl(y_i - \hat{y}(x_i)\bigr)^2 .$$

The first term does not contain the parameters and the second is the sum of
squared errors times a positive constant. So **minimising the Gaussian negative
log likelihood is minimising the mean squared error**, with the same minimiser.

### Your turn

Check it numerically: evaluate both objectives on a grid of $(a, b)$ and compare
where each is smallest.

In [ ]:
# two objectives, one minimiser ---------------------------------------------
def nll(a, b, sigma):
    residual = y - (a * x + b)
    n = len(x)
    return 0.5 * n * np.log(2 * np.pi * sigma ** 2) + np.sum(residual ** 2) / (2 * sigma ** 2)

def mse(a, b):
    return np.mean((y - (a * x + b)) ** 2)

a_grid = np.linspace(1.8, 3.0, 601)
b_grid = np.linspace(0.2, 1.4, 601)
NLL = np.array([[nll(a, b, core.TRUE_SIGMA) for b in b_grid] for a in a_grid])
MSE = np.array([[mse(a, b) for b in b_grid] for a in a_grid])

ia, ib = np.unravel_index(NLL.argmin(), NLL.shape)
a_nll, b_nll = a_grid[ia], b_grid[ib]
ia, ib = np.unravel_index(MSE.argmin(), MSE.shape)
a_mse, b_mse = a_grid[ia], b_grid[ib]
# ------------------------------------------------------------------------------

In [ ]:
# Least squares, solved directly.
#   np.stack([x, ones], axis=1) is the design matrix: one column for
#   the slope, one of ones for the intercept.
#   np.linalg.lstsq solves it in one step - no gradient descent, no
#   learning rate, and it is exact for a linear model under MSE.
# Two columns: the first multiplies the slope, the second - all
# ones - multiplies the intercept. That is how a bias becomes
# just another weight.
design = np.stack([x, np.ones_like(x)], axis=1)
a_ls, b_ls = np.linalg.lstsq(design, y, rcond=None)[0]

print(core.error_table(
    [["negative log likelihood (grid)", f"{a_nll:.4f}", f"{b_nll:.4f}"],
     ["mean squared error (grid)", f"{a_mse:.4f}", f"{b_mse:.4f}"],
     ["least squares (exact)", f"{a_ls:.4f}", f"{b_ls:.4f}"],
     ["the truth", f"{core.TRUE_SLOPE:.4f}", f"{core.TRUE_INTERCEPT:.4f}"]],
    ["objective", "slope", "intercept"]))

# The same rings in both panels. The NLL is a constant plus n/(2 sigma^2)
# times the MSE, so every MSE ring is also an NLL ring: choose the MSE
# levels and convert them. Squared spacing puts the rings evenly apart,
# like the contour lines of a bowl.
t = np.linspace(0, 1, 26)[1:] ** 2
mse_levels = MSE.min() + t * (MSE.max() - MSE.min())
nll_levels = (0.5 * len(x) * np.log(2 * np.pi * core.TRUE_SIGMA ** 2)
              + len(x) * mse_levels / (2 * core.TRUE_SIGMA ** 2))

fig, axes = plt.subplots(1, 2, figsize=(13.6, 4.6))
for ax, G, levels, name in zip(axes, [NLL, MSE], [nll_levels, mse_levels],
                               ["negative log likelihood", "mean squared error"]):
    cs = ax.contour(b_grid, a_grid, G, levels=levels, cmap="viridis")
    fig.colorbar(cs, ax=ax, label=name + " on each ring")   # the colour legend
    ax.plot(b_ls, a_ls, "x", ms=13, mew=2.4, color="#d94f2b",
            label="least squares")
    ax.plot(core.TRUE_INTERCEPT, core.TRUE_SLOPE, "o", ms=9, mfc="none",
            mec="#111111", mew=2.0, label="truth")
    ax.set_xlabel("intercept $b$"); ax.set_ylabel("slope $a$")
    ax.set_title(name); ax.legend(frameon=False, fontsize=9)
plt.show()

**What you should see.**

| objective | slope | intercept |
| --- | --- | --- |
| negative log likelihood (grid) | 2.3960 | 0.7860 |
| mean squared error (grid) | 2.3960 | 0.7860 |
| least squares (exact) | 2.3954 | 0.7871 |
| the truth | 2.4000 | 0.8000 |

and two contour plots with identical ellipses and identical centres.

The first two rows agree exactly; their gap to the exact least squares is the
grid spacing, 0.002. The gap to the truth is **estimation error**: forty noisy
readings do not pin down the true line, and only more data fixes that.

---

## 2 · From a categorical assumption to the cross entropy

**The goal.** Section 1 predicted a number. Many problems ask for a
**category** instead. The example in this notebook is condition monitoring: an
accelerometer on a machine gives two numbers, the vibration amplitude at the
running speed and the amplitude in a high-frequency band. From them we want to
tell three conditions apart: **0 balanced** (both small), **1 imbalance**
(large at running speed) and **2 bearing fault** (large in the band). A
machine's **label** is its true condition, known from inspecting it. It is the
answer the model is scored against, never something computed from the model.

The recipe of section 1, minus the log of the probability of what was
observed, gives the loss for categories too: the **cross entropy**. This
section writes it by hand and checks it against PyTorch.

**No network and no data yet.** The six rows of logits below are written by
hand, each to produce one scenario the loss has to get right: right and sure,
undecided, confidently wrong, and so on. Known inputs give known answers, so a
mistake in your code shows. Section 3 trains a real network on real
measurements.

The output is now one of $C$ classes. The network's last layer outputs one
number per class, the **logits** $z_c$: raw scores, any real number, and the
bigger the logit the more likely the class. They are not probabilities yet —
they can be negative and need not sum to one — and the softmax turns them
into probabilities,

$$p_c = \frac{e^{z_c}}{\sum_{k} e^{z_k}}.$$

The recipe gives minus the log of the probability of the observed label $t$,

$$\mathcal{L} = -\log p_t
= -z_t + \log\sum_k e^{z_k},$$

and averaged over the data that is the **cross entropy**.

Two details. **Subtract the largest logit before exponentiating**: $e^{z}$
overflows above about 88 in single precision, and the softmax does not change.
**`nn.CrossEntropyLoss` takes logits, not probabilities**: apply a softmax first
and it is applied twice, which trains slowly to a worse answer with no warning.

### Your turn

Implement the softmax and the cross entropy in NumPy, and check against PyTorch.

In [ ]:
# softmax and cross entropy, stably -----------------------------------------
def softmax_np(z):                 # z has shape (N, C)
    shifted = z - z.max(axis=1, keepdims=True)    # subtract the row max: no overflow in exp
    e = np.exp(shifted)
    return e / e.sum(axis=1, keepdims=True)

def cross_entropy_np(z, t):        # t has shape (N,), integer labels
    shifted = z - z.max(axis=1, keepdims=True)
    log_p   = shifted - np.log(np.exp(shifted).sum(axis=1, keepdims=True))
    return -log_p[np.arange(len(t)), t].mean()

logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.3],
                   [1.2, 0.7, 3.1],
                   [0.0, 0.0, 0.0],
                   [8.0, -2.0, -3.0],
                   [-1.0, -1.0, 9.0]])
labels = np.array([0, 1, 2, 1, 2, 2])

ce_numpy = float(cross_entropy_np(logits, labels))
ce_torch = float(nn.CrossEntropyLoss()(torch.tensor(logits), torch.tensor(labels)))
# ------------------------------------------------------------------------------

In [ ]:
# softmax turns scores into probabilities; cross entropy scores the
# probability given to the correct class. Both written by hand here,
# in the numerically stable form: subtract the row maximum before
# the exponential, or a large logit overflows.
probs = softmax_np(logits)
print("softmax rows sum to one:", np.allclose(probs.sum(axis=1), 1.0))
print()

# The six rows are not data. Each row of logits was written by hand to
# produce one scenario the loss has to get right; the last column names it.
# loss = -log(probability given to the true label); its mean over the six
# rows is the cross entropy printed below the table.
scenarios = ["right, fairly sure", "right, sure", "right, sure",
             "no preference: a third each", "confidently wrong",
             "confidently right"]
print(core.error_table(
    [[", ".join(f"{v:.1f}" for v in z), "  ".join(f"{p:.4f}" for p in row),
      str(t), f"{-np.log(row[t]):.4f}", name]
     for z, row, t, name in zip(logits, probs, labels, scenarios)],
    ["logits", "probabilities", "label", "loss", "scenario"]))
print()
print(f"cross entropy, NumPy : {ce_numpy:.10f}")
print(f"cross entropy, torch : {ce_torch:.10f}")
print(f"difference           : {abs(ce_numpy - ce_torch):.2e}")
print()
print("loss for a uniform three-class guess: ln 3 =", round(float(np.log(3)), 6))

**What you should see.** `softmax rows sum to one: True`, then one row per
scenario:

| logits | probabilities | label | loss | scenario |
| --- | --- | ---: | ---: | --- |
| 2.0, 1.0, 0.1 | 0.6590  0.2424  0.0986 | 0 | 0.4170 | right, fairly sure |
| 0.5, 2.5, 0.3 | 0.1086  0.8025  0.0889 | 1 | 0.2200 | right, sure |
| 1.2, 0.7, 3.1 | 0.1206  0.0731  0.8063 | 2 | 0.2153 | right, sure |
| 0.0, 0.0, 0.0 | 0.3333  0.3333  0.3333 | 1 | 1.0986 | no preference: a third each |
| 8.0, -2.0, -3.0 | 0.9999  0.0000  0.0000 | 2 | 11.0001 | confidently wrong |
| -1.0, -1.0, 9.0 | 0.0000  0.0000  0.9999 | 2 | 0.0001 | confidently right |

The zeros are rounding: the softmax never gives exactly 0, which is why row
five's loss is 11 and not infinite. The mean of the loss column is the cross
entropy,

```
cross entropy, NumPy : 2.1585311972
cross entropy, torch : 2.1585311972
difference           : 0.00e+00
```

or a difference around $10^{-16}$.

Row four has all logits zero, so every probability is one third and the loss is
$\ln 3 = 1.0986$, the number printed last. A three-class cross entropy that does
not start near 1.10 means the labels, the shapes or the initialisation are
wrong; for two classes it is $\ln 2 = 0.693$, for ten $\ln 10 = 2.303$. Row five,
logits `[8, -2, -3]` with label 2, is confidently wrong and costs about 11.

---

## 3 · The two losses on the same problem

**How this connects to sections 1 and 2.** The recipe is always the same, minus
the log of the probability of what was measured, but its answer depends on
what you assume. Gaussian noise on a number gives the MSE (section 1); a
category gives the cross entropy (section 2). In section 1 the NLL and the MSE
were **one loss written two ways**. Here the cross entropy and the MSE are
**two different losses**: the MSE on the predicted probabilities is what you
would write out of habit from regression. This section trains the same network
with each and measures what the habit costs.

**The data.** 360 simulated machines, 120 in each condition, each described by
its two vibration numbers. The classes overlap, as real ones do, so no model
gets every machine right. 240 machines train the network. The other 120 are
**held out**: never seen in training, they show how the model does on machines
it has not met.

**Two measures, and why both.** **Accuracy** is the share of held-out machines
whose most probable class is the true one: is the model right? The **held-out
cross entropy** asks how good the probabilities are: low when the model is
confident and right, high when it is confident and wrong. The second matters
whenever a decision uses the probability itself, such as an alarm that rings
at $p(\text{bearing fault}) > 0.8$.

**One-hot.** For the MSE, a label has to look like the three probabilities it
is compared with: label 2 becomes $[0, 0, 1]$. The cross entropy takes the
label as it is.

Train the same small network on the three vibration classes twice: with cross
entropy on the logits, and with mean squared error on the softmax outputs
against one-hot targets, which is what many people write when they come to
classification from regression.

### Your turn

In [ ]:
# the same network under two losses -----------------------------------------
X, y_cls = core.vibration_dataset()
X_train, y_train = X[:240], y_cls[:240]
X_test,  y_test  = X[240:], y_cls[240:]

Xa = torch.tensor(X_train); ya = torch.tensor(y_train)
Xb = torch.tensor(X_test)
Y_onehot = torch.tensor(core.one_hot(y_train).astype(np.float32))

def loss_ce(model):
    return nn.CrossEntropyLoss()(model(Xa), ya)

def loss_mse(model):
    return nn.MSELoss()(torch.softmax(model(Xa), dim=-1), Y_onehot)

results, models_cls = {}, {}
for name, loss_of in (("cross entropy", loss_ce), ("mean squared error", loss_mse)):
    core.set_seed(0)
    model = nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 3))
    optimiser = torch.optim.Adam(model.parameters(), lr=0.05)
    for epoch in range(400):
        optimiser.zero_grad()
        loss_of(model).backward()
        optimiser.step()
    with torch.no_grad():
        logits_test = model(Xb)
        acc  = float((logits_test.argmax(dim=1).numpy() == y_test).mean())
        held = float(nn.CrossEntropyLoss()(logits_test, torch.tensor(y_test)))   # the same yardstick for both
    results[name] = (acc, held)
    models_cls[name] = model
# ------------------------------------------------------------------------------

In [ ]:
# The same network trained under both losses: the same accuracy, but
# not the same quality of probabilities.
print(core.error_table(
    [[name, f"{acc:.3f}", f"{held:.4f}"] for name, (acc, held) in results.items()],
    ["training loss", "held-out accuracy", "held-out cross entropy"]))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.6))
for ax, name in zip(axes, results):
    with torch.no_grad():
        pred = models_cls[name](Xb).numpy().argmax(axis=1)
    core.plot_classes(X_test, y_test, ax=ax, predictions=pred,
                      title=f"trained with {name}")
plt.show()

**What you should see.**

| training loss | held-out accuracy | held-out cross entropy |
| --- | --- | --- |
| cross entropy | 0.967 | 0.1191 |
| mean squared error | 0.967 | 0.1687 |

**The accuracies are identical**: both losses put the boundaries between the
three conditions in the same places, and the few machines they get wrong sit
where the classes overlap. **The held-out cross entropies are not**: the model
trained with squared error is about forty per cent worse at the probabilities,
so its confidence is a worse guide to its accuracy. For an alarm set at
$p(\text{bearing fault}) > 0.8$, that difference decides when the alarm rings.

**Why.** Squared error is measured on the probability, and the softmax goes flat
when the model is confidently wrong, so there the gradient of the squared error
is close to zero: the worst mistakes are the ones it corrects least. The
gradient of the cross entropy with respect to the logits is $p - y$, the
probability still missing, and it stays large exactly there.

---

## 4 · Change the noise assumption, change the loss

**The goal.** Back to the load cell of section 1. The MSE assumes Gaussian noise,
in which every error is small. Real instruments sometimes produce a reading that
is simply wrong: a number typed wrongly, a loose connector. This section
corrupts one reading in forty and asks which loss still finds the true line.

**The mean absolute error (MAE).** The average size of the residuals, ignoring
their sign:
$$\mathrm{MAE} = \frac{1}{n}\sum_{i=1}^{n}\bigl|\,y_i - \hat{y}(x_i)\bigr| .$$
The MSE squares each residual, so one residual of 4 counts 1,600 times as much
as a residual of 0.1. The MAE counts it 40 times as much. In PyTorch it is
`nn.L1Loss`.

**Where the MAE comes from.** The same recipe with a different noise
assumption: **Laplace** noise instead of Gaussian. Like the Gaussian it peaks at
zero error, but its tails fall off more slowly, so a large error is rare rather
than practically impossible. The paragraph below does the derivation.

**The experiment.** One reading is raised by 4.0, about eleven times $\sigma$. The
line is fitted twice, once with each loss, by gradient descent (Adam). Section
1's exact formula cannot be used, because the MAE has none. Both fits are
compared with the true line.

A bad reading, a stuck bit or a knocked cable, is not Gaussian. A Gaussian gives
a residual of $10\sigma$ a probability of about $10^{-23}$, so squared error
moves the whole line to accommodate it. Assume Laplace noise instead,
$p(r) \propto e^{-|r|/b}$, where large errors are less rare, and the recipe gives
$-\log p \propto |r|$: the **mean absolute error**, which weights a large
residual only linearly.

### Your turn

Corrupt one reading and fit the same line under both losses.

In [ ]:
# one bad reading, two losses -----------------------------------------------
x_bad, y_bad = core.add_outlier(x, y, index=7, offset=4.0)
xb, yb = core.to_tensor(x_bad), core.to_tensor(y_bad)

fits = {}
for name, loss_fn in (("MSE", nn.MSELoss()), ("MAE", nn.L1Loss())):
    core.set_seed(0)
    line = nn.Linear(1, 1)
    optimiser = torch.optim.Adam(line.parameters(), lr=0.05)
    for epoch in range(3000):
        optimiser.zero_grad()
        loss = loss_fn(line(xb), yb)
        loss.backward()
        optimiser.step()
    fits[name] = (float(line.weight.item()), float(line.bias.item()))
# ------------------------------------------------------------------------------

In [ ]:
# One reading recorded wrongly (core.add_outlier) and two fits: the
# squared error chases the outlier, the absolute error mostly does not.
print(core.error_table(
    [[name, f"{a:.4f}", f"{b:.4f}",
      f"{abs(a - core.TRUE_SLOPE):.4f}", f"{abs(b - core.TRUE_INTERCEPT):.4f}"]
     for name, (a, b) in fits.items()]
    + [["truth", f"{core.TRUE_SLOPE:.4f}", f"{core.TRUE_INTERCEPT:.4f}",
        "-", "-"]],
    ["loss", "slope", "intercept", "slope error", "intercept error"]))

grid = np.linspace(0, 2, 100)
fig, ax = plt.subplots(figsize=(7.0, 4.4))
ax.plot(grid, core.TRUE_SLOPE * grid + core.TRUE_INTERCEPT, lw=1.6, ls="--",
        color="#888888", label="truth")
for i, (name, (a, b)) in enumerate(fits.items()):
    ax.plot(grid, a * grid + b, lw=2.0,
            color=["#d94f2b", "#1f77b4"][i], label=f"fitted with {name}")
ax.plot(x_bad, y_bad, "o", ms=6, color="#111111", label="readings")
ax.plot(x_bad[7], y_bad[7], "o", ms=13, mfc="none", mec="#d94f2b", mew=2.4,
        label="the bad reading")
ax.set_xlabel("applied load"); ax.set_ylabel("sensor reading")
ax.set_title("One bad reading in forty")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.**

| loss | slope | intercept | slope error | intercept error |
| --- | --- | --- | --- | --- |
| MSE | 2.1824 | 1.0762 | 0.2176 | 0.2762 |
| MAE | 2.3616 | 0.8086 | 0.0384 | 0.0086 |
| truth | 2.4000 | 0.8000 | - | - |

and a figure in which the red MSE line is dragged towards the circled point
while the blue MAE line ignores it.

One bad reading in forty moved the least-squares slope by nine per cent and the
intercept by thirty-five; the absolute-error fit stays within two per cent of
the truth. The absolute error is not free: it is not differentiable at zero,
converges more slowly, and wastes information when the noise really is
Gaussian. The **Huber** loss, `nn.SmoothL1Loss`, is quadratic for small
residuals and linear for large ones, and is what you would use on instrument
data.

---

## 5 · Save

**What this does.** It writes this notebook's main numbers (the fitted lines,
the cross-entropy check, the two classifiers' accuracies and cross entropies,
the two outlier fits) to a file. The report in notebook 05 reads them, so run
it once before moving on. There is nothing to change here.

In [ ]:
# Saved for the report in notebook 05.
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb01_losses.npz")
np.savez(path,
         a_nll=a_nll, b_nll=b_nll, a_mse=a_mse, b_mse=b_mse,
         a_ls=a_ls, b_ls=b_ls,
         ce_numpy=ce_numpy, ce_torch=ce_torch,
         acc_ce=results["cross entropy"][0],
         acc_mse=results["mean squared error"][0],
         heldout_ce_ce=results["cross entropy"][1],
         heldout_ce_mse=results["mean squared error"][1],
         fit_mse=np.asarray(fits["MSE"]), fit_mae=np.asarray(fits["MAE"]))
print("wrote", path)
core.saved(path)


**What you should see.** `wrote .../Ex06_outputs/nb01_losses.npz`.

---

## 6 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The negative log likelihood and the mean squared error had the same
   minimiser. Using this as the example, say where a loss function comes from —
   the steps from choosing a distribution to minimising the negative log
   likelihood — and give one thing you can do with the first that you cannot do
   with the second.
   *→ L6.1 Q1, Q2*
2. Section 1 used the load cell's $\sigma$ from its calibration, and the fitted
   line did not depend on it. What distribution does squared error assume, and
   why does the value of $\sigma$ not move the minimiser? What changes when the
   network predicts $\sigma$ as well as $\mu$, as in L6.1?
   *→ L6.1 Q3*
3. Both losses gave the same accuracy on the vibration data but different
   held-out cross entropies. Where does cross entropy come from, and what should
   an untrained model report here, with three classes — and with ten? Describe an
   application in which you would not care about the difference, and one in
   which you would refuse to deploy the worse-calibrated model.
   *→ L6.1 Q5*
4. You are given a dataset in which about one reading in fifty is a
   transcription error, and the rest are Gaussian. Write down the loss you would
   use and the assumption it corresponds to, and say what it costs you if the
   noise was Gaussian after all.
   *→ L6.1 Q4*

*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 02](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_02_optimiser_comparison_light.ipynb)**, where a network is trained with Adam, then
L-BFGS: the recipe every exercise in Part 2 uses, measured rather than
asserted.
